# Practical Q6 - Artificial Neural Network
## 6A: Regression Model | 6B: Classification Model

---
# 6A: REGRESSION MODEL (Housing Price Prediction)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn import metrics

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

X = housing.data
Y = housing.target

print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)
print("Feature names:", housing.feature_names)

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
print("Data normalized.")

In [ ]:
# ── Train-Test Split Evaluation ──

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

regModel = MLPRegressor(hidden_layer_sizes=(256, 128, 64, 32),
                        activation='relu',
                        solver='sgd',
                        max_iter=1000,
                        random_state=42)

regModel.fit(X_train, Y_train)

Y_pred = regModel.predict(X_test)

mse  = metrics.mean_squared_error(Y_test, Y_pred)
r2   = metrics.r2_score(Y_test, Y_pred)

print("Train-Test Split Results")
print("MSE    :", round(mse, 4))
print("R2 Score:", round(r2, 4))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(Y_test, Y_pred, alpha=0.4, color='steelblue', edgecolors='k', linewidths=0.3)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.title('ANN Regressor: Actual vs Predicted (Train-Test Split)')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── K-Fold Cross Validation ──

kf = KFold(n_splits=10, shuffle=True, random_state=42)

mse_scores = []
r2_scores  = []

for fold, (train_index, test_index) in enumerate(kf.split(X), 1):

    print("\n")
    print("Fold:", fold)

    X_train = X[train_index]
    X_test  = X[test_index]
    Y_train = Y[train_index]
    Y_test  = Y[test_index]

    regModel = MLPRegressor(hidden_layer_sizes=(256, 128, 64, 32),
                            activation='relu',
                            solver='sgd',
                            max_iter=1000,
                            random_state=42)

    regModel.fit(X_train, Y_train)

    Y_pred = regModel.predict(X_test)

    mse = metrics.mean_squared_error(Y_test, Y_pred)
    r2  = metrics.r2_score(Y_test, Y_pred)

    print("MSE     :", round(mse, 4))
    print("R2 Score:", round(r2, 4))

    mse_scores.append(mse)
    r2_scores.append(r2)

print("\nFINAL")
print("Average MSE     :", round(np.mean(mse_scores), 4))
print("Average R2 Score:", round(np.mean(r2_scores), 4))

---
# 6B: CLASSIFICATION MODEL (Iris - Multi-class)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from mlxtend.plotting import plot_confusion_matrix

In [ ]:
df = sns.load_dataset('iris')
df.head()

In [ ]:
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
Y = df['species'].values

print("Shape of X:", X.shape)
print("Classes   :", np.unique(Y))

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
print("Data normalized.")

In [ ]:
def evaluate_model(model, X, Y):

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=2)

    acc_scores       = []
    precision_scores = []
    recall_scores    = []
    f1_scores        = []

    for fold, (train_index, test_index) in enumerate(skf.split(X, Y), 1):

        print("\n")
        print("Fold:", fold)

        X_train = X[train_index]
        X_test  = X[test_index]
        Y_train = Y[train_index]
        Y_test  = Y[test_index]

        model.fit(X_train, Y_train)

        Y_testPred  = model.predict(X_test)
        Y_trainPred = model.predict(X_train)

        train_acc = metrics.accuracy_score(Y_train, Y_trainPred)
        test_acc  = metrics.accuracy_score(Y_test,  Y_testPred)

        precision = metrics.precision_score(Y_test, Y_testPred, average='macro')
        recall    = metrics.recall_score(Y_test, Y_testPred, average='macro')
        f1        = metrics.f1_score(Y_test, Y_testPred, average='macro')

        print("Train Accuracy :", round(train_acc, 4))
        print("Test  Accuracy :", round(test_acc,  4))
        print("Precision      :", round(precision, 4))
        print("Recall         :", round(recall,    4))
        print("F1 Score       :", round(f1,        4))

        acc_scores.append(test_acc)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)

        cm = metrics.confusion_matrix(Y_test, Y_testPred)

        print("\nConfusion Matrix:")
        plot_confusion_matrix(conf_mat=cm)
        plt.title(f"Confusion Matrix - Fold {fold}")
        plt.show()

        print("\nClassification Report:")
        print(metrics.classification_report(Y_test, Y_testPred))

    print("FINAL")
    print("Average Accuracy  :", round(np.mean(acc_scores),       4))
    print("Average Precision :", round(np.mean(precision_scores), 4))
    print("Average Recall    :", round(np.mean(recall_scores),    4))
    print("Average F1 Score  :", round(np.mean(f1_scores),        4))

In [ ]:
print("\n ANN CLASSIFIER")

clsModel = MLPClassifier(hidden_layer_sizes=(34, 22),
                         activation='relu',
                         solver='sgd',
                         max_iter=1000,
                         random_state=42)

evaluate_model(clsModel, X, Y)